In [1]:
import pyranges as pr
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
BASE = Path("../data/alzheimer-lrseq")
BASE_GTF = Path("../.data")

SAMPLES = pd.DataFrame([
    {"sample_id": "alzheimer",  "condition": "Alzheimer", "age": 90},
    {"sample_id": "alzheimer2", "condition": "Alzheimer", "age": 86},
    {"sample_id": "healthy",    "condition": "Healthy",   "age": 90},
    {"sample_id": "healthy2",   "condition": "Healthy",   "age": 85},
])

MOD_TYPE_MAP = {
    "a": "m6A", "m": "m5C",
    "17802": "Pseudouridine", "17596": "Inosine",
    "19227": "Nm", "19228": "Nm", "19229": "Nm", "69426": "Nm",
}

BED_COLS = ["chrom","start","end","mod_code","score","strand",
            "thick_start","thick_end","color","coverage","mod_freq",
            "mod_count","unmod_count","v14","v15","v16","v17","v18"]

In [3]:
def load_exons(sample_id):
    gtf_path = BASE_GTF / "stringtie" / sample_id / "transcripts_filtered.gtf"
    gtf = pr.read_gtf(str(gtf_path))
    df = gtf[gtf.Feature == "exon"].df
    df = df[df["Strand"].isin(["+", "-"])].copy()
    df = df.reset_index(drop=True)
    return pr.PyRanges(df[["Chromosome", "Start", "End", "Strand", "transcript_id"]])

exons_by_sample = {}
for sid in SAMPLES.sample_id:
    exons_by_sample[sid] = load_exons(sid)
    print(f"  {len(exons_by_sample[sid])} exons")

  267139 exons
  435652 exons
  494993 exons
  364018 exons


In [4]:
def load_bed(sample_id):
    bed_dir = BASE / "modifications" / sample_id / "bed"
    dfs = []
    for f in sorted(bed_dir.glob("*.bed.gz")):
        df = pd.read_csv(
            f, sep = "\t", header = None,
            names = BED_COLS, usecols = ["chrom","start","end","mod_code","mod_freq","coverage","strand"]
        )
        dfs.append(df)
    bed = pd.concat(dfs, ignore_index = True)
    bed["mod_type"] = bed["mod_code"].astype(str).map(MOD_TYPE_MAP).fillna(bed["mod_code"].astype(str))
    return bed

bed_by_sample = {}
for sid in SAMPLES.sample_id:
    bed_by_sample[sid] = load_bed(sid)
    print(f"  {len(bed_by_sample[sid]):,} modification sites")
    print(bed_by_sample[sid].mod_type.value_counts().to_string())

  348,274 modification sites
mod_type
m6A              197201
m5C               58487
Nm                56302
Inosine           19622
Pseudouridine     16662
  696,986 modification sites
mod_type
m6A              378117
m5C              122510
Nm               116125
Inosine           46663
Pseudouridine     33571
  661,266 modification sites
mod_type
m6A              331682
m5C              133123
Nm               115724
Inosine           44176
Pseudouridine     36561
  373,790 modification sites
mod_type
m6A              197990
m5C               74331
Nm                59408
Inosine           21226
Pseudouridine     20835


In [5]:
def overlap_sample(sample_id):
    exons_pr = exons_by_sample[sample_id]
    bed = bed_by_sample[sample_id]
    bed_pr = pr.PyRanges(bed.rename(columns={
        "chrom": "Chromosome", "start": "Start", "end": "End", "strand": "Strand"
    }))
    joined = exons_pr.join(bed_pr, strandedness=False)
    df = joined.df
    df = df[df["Strand"] == df["Strand_b"]]
    return df[["transcript_id", "Strand", "Start_b", "mod_type", "mod_freq", "coverage"]]

overlaps = {}
for sid in SAMPLES.sample_id:
    print(f"Overlapping: {sid}")
    overlaps[sid] = overlap_sample(sid)
    print(f"  {len(overlaps[sid]):,} modification-exon hits")

Overlapping: alzheimer


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


  748,353 modification-exon hits
Overlapping: alzheimer2


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


  1,800,980 modification-exon hits
Overlapping: healthy


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


  1,873,962 modification-exon hits
Overlapping: healthy2
  909,973 modification-exon hits


join: Strand data from other will be added as strand data to self.
If this is undesired use the flag apply_strand_suffix=False.
To turn off the warning set apply_strand_suffix to True or False.


In [6]:
SQANTI_COLS = [
    "isoform", "chrom", "strand", "start", "end", "length", "exons",
    "structural_category", "associated_gene",
    "diff_to_TSS", "diff_to_TTS",
    "FL", "iso_exp", "gene_exp", "ratio_exp",
    "coding", "predicted_NMD",
    "perc_A_downstream_TTS",
    "dist_to_CAGE_peak", "within_CAGE_peak",
    "dist_to_polyA_site", "within_polyA_site",
    "polyA_motif_found",
]

sqanti = {}
for sid in SAMPLES.sample_id:
    path = BASE / "sqanti" / sid / f"{sid}_classification.txt"
    df = pd.read_csv(path, sep="\t", usecols=SQANTI_COLS, low_memory=False)
    df = df.rename(columns={"isoform": "transcript_id"})
    sqanti[sid] = df
    print(f"  {sid}: {len(sqanti[sid]):,} isoforms")

  alzheimer: 39,911 isoforms
  alzheimer2: 49,663 isoforms
  healthy: 57,315 isoforms
  healthy2: 41,071 isoforms


In [7]:
def aggregate_mods(df, tx_meta):
    df = df.merge(
        tx_meta[["transcript_id", "start", "end", "strand", "length"]],
        on="transcript_id", how="left"
    )
    df["rel_pos"] = np.where(
        df["strand"] == "+",
        (df["Start_b"] - df["start"]) / df["length"],
        (df["end"] - df["Start_b"]) / df["length"],
    ).clip(0, 1)
    df["weighted_freq"] = df["mod_freq"] * df["coverage"]

    agg = df.groupby(["transcript_id", "mod_type"]).agg(
        cov_sum=("coverage", "sum"),
        wf_sum=("weighted_freq", "sum"),
        n_sites=("mod_freq", "count"),
        mean_pos=("rel_pos", "mean"),
        std_pos=("rel_pos", "std"),
    ).reset_index()
    agg["mean_freq"] = agg["wf_sum"] / agg["cov_sum"]
    agg["std_pos"] = agg["std_pos"].fillna(0)

    def pivot_col(col, suffix):
        wide = agg.pivot(index="transcript_id", columns="mod_type", values=col).fillna(0)
        wide.columns = [f"{c}_{suffix}" for c in wide.columns]
        return wide

    return pd.concat([
        pivot_col("mean_freq", "freq"),
        pivot_col("n_sites", "n_sites"),
        pivot_col("mean_pos", "mean_pos"),
        pivot_col("std_pos", "std_pos"),
    ], axis=1).reset_index()

mod_features = {}
for sid in SAMPLES.sample_id:
    mod_features[sid] = aggregate_mods(overlaps[sid], sqanti[sid])
    print(f"  {sid}: {len(mod_features[sid]):,} transcripts")

  alzheimer: 27,667 transcripts
  alzheimer2: 39,877 transcripts
  healthy: 45,838 transcripts
  healthy2: 32,086 transcripts


In [8]:
parts = []
for sid in SAMPLES.sample_id:
    merged = sqanti[sid].merge(mod_features[sid], on="transcript_id", how="left")
    meta = SAMPLES.set_index("sample_id").loc[sid]
    merged["condition"] = meta["condition"]
    merged["age"] = meta["age"]
    merged["sample_id"] = sid
    parts.append(merged)

df_all = pd.concat(parts, ignore_index=True)

mod_cols = [c for c in df_all.columns if any(c.endswith(s) for s in ["_freq", "_n_sites", "_mean_pos", "_std_pos"])]
df_all[mod_cols] = df_all[mod_cols].fillna(0)

print(df_all.shape)
print(df_all["condition"].value_counts())
print(df_all["structural_category"].value_counts())

out_path = BASE / "ml_dataset.parquet"
df_all.to_parquet(out_path, index=False)
print(f"\nSaved: {out_path}")

(187960, 46)
condition
Healthy      98386
Alzheimer    89574
Name: count, dtype: int64
structural_category
full-splice_match          101127
novel_in_catalog            38995
incomplete-splice_match     20469
novel_not_in_catalog         9163
genic_intron                 8572
genic                        5419
antisense                    2477
fusion                       1169
intergenic                    569
Name: count, dtype: int64

Saved: ../data/alzheimer-lrseq/ml_dataset.parquet
